# Stage 3 — Decisions and price

**Checkpoint:** `stage-3`

Two apps on top of the one score: **adjudication** (do we say yes?) and **pricing** (at what
rate is yes worth it?). Notice that pricing contains **no machine learning at all** — and finds
the biggest number in the whole platform.

In [ ]:
# --- bootstrap: find the repo root, make the repo importable ---------------
# The modules use paths relative to the repo root (e.g. Path("score/models")),
# so we chdir there. This works whether you launched Jupyter from the repo
# root or from notebooks/.
import os, sys
from pathlib import Path

here = Path.cwd()
root = next((p for p in [here, *here.parents] if (p / "verify.py").exists()), None)
assert root is not None, "Could not find the repo root (no verify.py above cwd)."
os.chdir(root)
sys.path.insert(0, str(root))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 60)
pd.set_option("display.float_format", lambda v: f"{v:,.4f}")
plt.rcParams.update({"figure.figsize": (9, 4.2), "axes.spines.top": False,
                     "axes.spines.right": False, "axes.grid": True,
                     "grid.alpha": .25, "figure.dpi": 110})

print(f"repo root: {root}")
print(f"stage.txt: {(root / 'stage.txt').read_text().strip()}")

In [ ]:
# --- stage guard: fail loudly and usefully, not mysteriously ---------------
NEEDS = [
    "adjudication/models/adjudication_model.pkl",
    "adjudication/models/policy_config.json",
]
missing = [p for p in NEEDS if not Path(p).exists()]
if missing:
    raise SystemExit(
        "This notebook needs stage-3 artifacts. Missing:\n  "
        + "\n  ".join(missing)
        + "\n\nYou are at stage " + (Path("stage.txt").read_text().strip())
        + ". Fix with either:\n"
        "  git checkout stage-3      # jump to the finished stage, or\n"
        "  make <the stage's build step>  # build it yourself (see the lab sheet)"
    )
print("stage-3 artifacts present.")

## 1. The decision mix

In [ ]:
import json, joblib
from shared.config import RAW
from adjudication.src.feature_engineering import (
    ADJ_FEATURE_COLUMNS, compute_adjudication_features)
from adjudication.src.policy import PolicyConfig, decide
from score.src.predict import predict_score_pd

businesses = pd.read_parquet(RAW / "businesses.parquet")
model  = joblib.load("adjudication/models/adjudication_model.pkl")
config = PolicyConfig.from_dict(json.loads(Path("adjudication/models/policy_config.json").read_text()))

X_all  = compute_adjudication_features(businesses)
pd_hat = model.predict_proba(X_all[ADJ_FEATURE_COLUMNS])[:, 1]
dec    = decide(X_all, pd_hat, config)
dec["score_band"] = predict_score_pd(businesses)["score_band"].to_numpy()

mix = dec["decision"].value_counts().reindex(["Approve", "Refer", "Decline"]).fillna(0)
COL = {"Approve": "#2f7d4f", "Refer": "#c9a227", "Decline": "#b3372e"}

fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))
axes[0].bar(mix.index, mix.values, color=[COL[k] for k in mix.index])
for i, v in enumerate(mix.values):
    axes[0].text(i, v + 60, f"{int(v):,}\n{v/len(dec):.1%}", ha="center", weight="bold")
axes[0].set_title(f"Decisions across {len(dec):,} applicants"); axes[0].set_ylabel("applicants")

ct = (pd.crosstab(dec["score_band"], dec["decision"], normalize="index")
        .reindex(["D", "C", "B", "A", "AAA"]).dropna(how="all"))
ct = ct[[c for c in ["Approve", "Refer", "Decline"] if c in ct.columns]]
ct.plot(kind="barh", stacked=True, ax=axes[1], color=[COL[c] for c in ct.columns], width=.75)
axes[1].set_title("Decision mix by score band"); axes[1].set_xlabel("share")
axes[1].legend(loc="lower right")
plt.tight_layout(); plt.show()

## 2. Why people were referred or declined

A model gives a probability. A **policy layer** turns it into a decision you can explain in a
letter — knockouts, PD zones, and named overrides.

In [ ]:
from collections import Counter
reasons = Counter(r for rs in dec["decision_reasons"] for r in (rs if isinstance(rs, list) else [rs]) if r)
top = pd.Series(dict(reasons.most_common(10))).sort_values()

fig, ax = plt.subplots(figsize=(9, 4.4))
ax.barh(top.index, top.values, color="#5b8ac6")
ax.set_xlabel("applicants"); ax.set_title("Most common decision reasons")
plt.tight_layout(); plt.show()
top.sort_values(ascending=False).to_frame("applicants")

## 3. Pricing — arithmetic, not machine learning

`recommended_rate = hurdle-clearing rate + margin`, where the hurdle-clearing rate is built from
cost of funds, expected loss (PD × LGD), opex and fees. Every term is auditable.

In [ ]:
from pricing.src.portfolio import price_population, portfolio_summary
from pricing.src.engine import (MarketAssumptions, break_even_rate,
                                hurdle_clearing_rate, recommended_rate, profit_waterfall)
from shared.config import MARKET

priced = price_population()
summary = portfolio_summary(priced)

print(f"booked loans priced      : {summary['n']:,}")
print(f"clear the ROE hurdle     : {summary['n_clears']:,}  ({summary['share_clears']:.1%})")
print(f"median ROE at quoted rate: {summary['median_roe']:.2%}")
print(f"MISPRICED EXPOSURE       : ${summary['mispriced_ead']:,.0f}")

In [ ]:
m = MarketAssumptions.from_market(MARKET)
pds = np.linspace(0.005, 0.25, 120)
ead = float(priced["ead"].median())

fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))
axes[0].plot(pds * 100, [break_even_rate(p, ead, m) * 100 for p in pds], label="break-even")
axes[0].plot(pds * 100, [hurdle_clearing_rate(p, ead, m) * 100 for p in pds], label="clears hurdle")
axes[0].plot(pds * 100, [recommended_rate(p, ead, m) * 100 for p in pds], label="recommended", lw=2.5)
axes[0].set_xlabel("PD (%)"); axes[0].set_ylabel("rate (%)")
axes[0].set_title("Price is a function of risk"); axes[0].legend()

axes[1].hist(priced["roe_at_quoted"] * 100, bins=50, color="#9fb4d4", edgecolor="white")
axes[1].axvline(m.roe_hurdle * 100, color="#b3372e", ls="--", lw=2,
                label=f"hurdle {m.roe_hurdle:.0%}")
axes[1].set_xlabel("ROE at the quoted rate (%)"); axes[1].set_title("Who actually earns their cost of capital?")
axes[1].legend()
plt.tight_layout(); plt.show()

### The $1.28B finding

The largest result in this platform is **arithmetic on top of the score** — loans booked below
the rate their risk deserves. No better model would have found it, because it was never a
modelling problem.

In [ ]:
mis = priced[priced["mispriced"]]
by_band = (priced.groupby("score_band", observed=True)
           .apply(lambda g: pd.Series({
               "loans": len(g),
               "mispriced_loans": int(g["mispriced"].sum()),
               "mispriced_ead": float(g.loc[g["mispriced"], "ead"].sum())}),
                  include_groups=False)
           .reindex(["D", "C", "B", "A", "AAA"]).dropna(how="all"))

fig, ax = plt.subplots(figsize=(9, 4))
ax.bar(by_band.index, by_band["mispriced_ead"] / 1e6, color="#b3372e")
ax.set_ylabel("mispriced exposure ($m)"); ax.set_xlabel("score band")
ax.set_title(f"Mispriced exposure by band — ${summary['mispriced_ead']/1e9:.2f}B total")
plt.tight_layout(); plt.show()

by_band.style.format({"loans": "{:,.0f}", "mispriced_loans": "{:,.0f}", "mispriced_ead": "${:,.0f}"})

### One loan, all the way through — the profit waterfall

In [ ]:
row = priced.sort_values("ead", ascending=False).iloc[0]
w = profit_waterfall(float(row["pd"]), float(row["ead"]), float(row["quoted_rate"]), m)
pd.Series(w).to_frame(f"{row['business_id']}  (EAD ${row['ead']:,.0f}, PD {row['pd']:.2%})")

---
**Next:** `04_stage4_portfolio_watch.ipynb` — stop looking at applicants, start managing the
book you already have.